# DM_G4_P0008_set_covering_facility

## 0. 학습 범위
- Gate: G4
- Phase: P0008
- Topic: Ch.6 정수계획 2주차 — set covering, facility location, coverage matrix, max coverage, set partitioning, set packing
- Goal: coverage matrix에서 `Ax>=1`, `Ax=1`, `Ax<=1`의 의미 차이를 설명하고 facility location 변수분리를 수행한다.
- Source basis: DM_PDF08 우선, DM_PDF04/DM_PDF03/DM_PDF06 보조
- 웹 근거 사용: 아니오
- Code mode: hint_only


## 1. 작성 규칙
- 풀이용 노트북에는 최종 선택 조합, 최종 목적값, 완성 풀이를 쓰지 않는다.
- 각 문제는 먼저 변수의 의미와 domain을 분리해서 쓴 뒤 목적함수와 제약식을 작성한다.
- Solver 설명은 변수셀, 목표셀, 제약식 좌변 셀, 우변 셀, int/bin 옵션을 구분해서 적는다.
- 반올림, IF 함수, 자동 0-1 선택 같은 지름길은 정답으로 인정하지 않는다.
- 답안은 각 문제의 빈 답안 템플릿에 작성한다.


## 2. 채점 기준
| 항목 | 배점 | 확인 기준 |
|---|---:|---|
| coverage matrix와 0-1 변수 정의 | 20 | 행은 수요구역, 열은 후보지임을 구분한다. |
| set covering/partitioning/packing 부등호 해석 | 20 | `>=1`, `=1`, `<=1`의 현실 의미를 설명한다. |
| max coverage x/y 변수 분리와 중복계산 방지 | 25 | 설치 변수와 혜택 변수를 분리하고 `y_i <= sum A_ij x_j`를 쓴다. |
| capacitated facility location formulation | 25 | assignment, open, capacity, availability를 모두 쓴다. |
| Solver mapping, B&B 연결, source anchor/node_id | 10 | matrix 셀, 변수셀, coverage count 셀, 목표셀을 연결한다. |


## 3. Set Covering 기준표
| 모형 | 핵심 제약 | 의미 |
|---|---|---|
| set covering | `sum_j A_ij x_j >= 1` | 수요구역 `i`가 적어도 하나의 선택 후보지로 덮인다. |
| set partitioning | `sum_j A_ij x_j = 1` | 정확히 하나에만 배정/분할된다. |
| set packing | `sum_j A_ij x_j <= 1` | 겹치지 않게 선택한다. |
| max coverage | `y_i <= sum_j A_ij x_j` | 커버되면 혜택 `y_i=1`, 여러 번 커버되어도 한 번만 계산한다. |
| capacitated facility location | `x_ij <= y_j`, `sum_i d_i x_ij <= cap_j y_j` | 열린 시설에만 배정하고 용량을 넘기지 않는다. |


## 4. 문제 세트

### 문제 1. 야간 응급 AED 설치 — set covering 최소 설치 모형

한 신도시가 야간 AED 보관함 후보지 6곳 중 일부를 설치하려고 한다. 수요구역은 8개다. 후보지 `j`가 수요구역 `i`를 6분 이내 커버하면 `A_ij=1`, 아니면 0이다.

Coverage matrix:

| 수요구역 i / 후보지 j | 1 | 2 | 3 | 4 | 5 | 6 |
|---:|---:|---:|---:|---:|---:|---:|
| 1 | 1 | 0 | 1 | 0 | 0 | 0 |
| 2 | 1 | 1 | 0 | 0 | 0 | 0 |
| 3 | 0 | 1 | 1 | 0 | 1 | 0 |
| 4 | 0 | 0 | 1 | 1 | 0 | 0 |
| 5 | 0 | 0 | 0 | 1 | 1 | 1 |
| 6 | 0 | 1 | 0 | 0 | 1 | 0 |
| 7 | 0 | 0 | 0 | 1 | 0 | 1 |
| 8 | 1 | 0 | 0 | 0 | 0 | 1 |

설치비용:

| 후보지 j | 1 | 2 | 3 | 4 | 5 | 6 |
|---:|---:|---:|---:|---:|---:|---:|
| 비용 | 8 | 7 | 6 | 9 | 5 | 6 |

요구:
1. `x_j`를 정의하라.
2. 모든 후보지가 동일 비용일 때 목적함수를 쓰라.
3. 설치비용을 고려할 때 목적함수를 쓰라.
4. 각 수요구역별 covering constraint를 전부 쓰라.
5. 일반형 `Ax >= 1`로 표현하라.
6. `Ax >= 1`에서 우변의 1이 무엇을 의미하는지 설명하라.
7. Solver에서 coverage matrix, variable cells, coverage count cells, objective cell을 설명하라.
8. 이 문제가 0-1 정수계획인지, assignment인지, transportation인지 판정하라.
9. set covering, set partitioning, set packing으로 바꿀 때 부등호가 각각 어떻게 달라지는지 설명하라.
10. 오답 진단: “각 수요구역은 정확히 하나의 AED에만 커버되어야 하므로 모든 제약은 =1이어야 한다.” 이 답안이 covering 문제에서 왜 틀릴 수 있는지 설명하라.
11. 복수 최적해가 발생할 수 있는 이유를 설명하라.

node_id: `n_DM_PDF08.set_covering_model`, `n_DM_PDF08.facility_location`, `n_DM_PDF08.coverage_matrix`, `n_DM_PDF08.set_covering_partitioning_packing`, `n_DM_PDF08.multiple_optima_facility`

source anchors: `DM_PDF08:p001:L002`, `DM_PDF08:p001:L004`, `DM_PDF08:p002:L003`, `DM_PDF08:p002:L005`, `DM_PDF08:p002:L007`, `DM_PDF08:p004:L003`, `DM_PDF08:p004:L007`, `DM_PDF08:p004:L008`, `DM_PDF08:p006:L003`, `DM_PDF08:p006:L006`, `DM_PDF08:p006:L007`, `DM_PDF08:p006:L008`, `DM_PDF08:p007:L009`


In [ ]:
# 힌트:
# - x_j = 후보지 j에 시설을 설치하면 1, 아니면 0이다.
# - A_ij = 수요구역 i가 후보지 j로 커버되면 1, 아니면 0이다.
# - set covering은 각 i에 대해 sum_j A_ij*x_j >= 1이다.
# - set partitioning은 각 i에 대해 sum_j A_ij*x_j = 1이다.
# - set packing은 각 i에 대해 sum_j A_ij*x_j <= 1이다.
# - max coverage에서는 시설 설치 변수 x_j와 혜택 여부 변수 y_i를 분리한다.
# - y_i <= sum_j A_ij*x_j 로 중복계산을 방지한다.


#### 문항별 시각화 학습자료 — 문제 1
- 목적: coverage matrix의 행과 열, `Ax>=1`의 coverage count를 직접 확인한다.
- 체크포인트: 동일비용 목적함수와 설치비용 목적함수는 같은 matrix를 쓰지만 목표셀이 다르다.
- 풀이용이므로 최적 후보지 조합은 표시하지 않는다.


In [ ]:
# 시각화 업데이트: 문제 1 coverage matrix와 Ax 계산 scaffold
import numpy as np
import matplotlib.pyplot as plt

A = np.array([
    [1,0,1,0,0,0],
    [1,1,0,0,0,0],
    [0,1,1,0,1,0],
    [0,0,1,1,0,0],
    [0,0,0,1,1,1],
    [0,1,0,0,1,0],
    [0,0,0,1,0,1],
    [1,0,0,0,0,1],
])
cost = np.array([8,7,6,9,5,6])
# 학생이 실험할 후보 조합을 직접 바꾸어 본다. 예: np.array([1,0,0,1,1,0])
trial_x = np.array([0,0,0,0,0,0])
coverage_count = A @ trial_x

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
ax = axes[0]
im = ax.imshow(A, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(6), labels=[f"C{j}" for j in range(1,7)], rotation=45)
ax.set_yticks(range(8), labels=[f"Zone{i}" for i in range(1,9)])
for i in range(A.shape[0]):
    for j in range(A.shape[1]):
        ax.text(j, i, str(A[i,j]), ha="center", va="center", color="black")
ax.set_title("Coverage matrix A")

axes[1].bar(range(1,7), cost, color="#8ab6d6")
axes[1].set_xticks(range(1,7))
axes[1].set_xlabel("candidate")
axes[1].set_ylabel("install cost")
axes[1].set_title("install cost coefficients")

axes[2].bar(range(1,9), coverage_count, color=["#6aa84f" if c>=1 else "#d9d9d9" for c in coverage_count])
axes[2].axhline(1, color="#d84a4a", ls="--", label="covering RHS=1")
axes[2].set_xticks(range(1,9))
axes[2].set_xlabel("demand zone")
axes[2].set_ylabel("coverage count")
axes[2].set_title("Ax for trial_x")
axes[2].legend(fontsize=8)
plt.tight_layout()
plt.show()

print("현재 trial_x:", trial_x.tolist())
print("설치 수 목적값:", int(trial_x.sum()), " / 설치비용 목적값:", int(cost @ trial_x))
print("체크: 모든 coverage count가 1 이상이면 Ax>=1을 만족한다.")


### 내 답안
- 변수 정의:
- 목적함수:
- 제약식:
- domain:
- Solver mapping:
- LP relaxation/논리 해석:
- 오답 진단:
- source anchor / node_id:


### 문제 2. 이동검진 버스 배치 — max coverage와 중복계산 오답진단

한 보건소가 이동검진 버스 후보지 6곳 중 최대 3곳을 운영한다. 모든 구역을 반드시 커버할 수는 없으므로, 커버되는 주민 수를 최대화하려고 한다.

수요구역 인구:

| 구역 i | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 |
|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| 인구 | 5.0 | 7.5 | 4.0 | 6.5 | 8.0 | 3.5 | 9.0 | 4.5 |

Coverage matrix는 문제 1과 같다.

추가 조건:
1. 최대 3개 후보지만 운영할 수 있다.
2. 후보지 1과 4는 같은 주차장 계약을 사용하므로 동시에 선택할 수 없다.
3. 후보지 6을 선택하려면 후보지 4도 선택해야 한다.
4. 최소 한 개의 동부권 후보지 4, 5, 6 중 하나는 선택해야 한다.

요구:
1. 시설 설치 변수 `x_j`를 정의하라.
2. 수요구역 혜택 여부 변수 `y_i`를 정의하라.
3. 왜 max coverage에서는 `x_j`만으로 목적함수를 쓰면 중복계산 위험이 있는지 설명하라.
4. objective function `Maximize sum_i population_i*y_i`를 쓰라.
5. 각 구역에 대해 `y_i <= sum_j A_ij x_j` 제약을 쓰라.
6. `y_i`의 domain을 쓰라.
7. 최대 3개 후보지 조건을 쓰라.
8. 후보지 1과 4의 상호배타 조건을 쓰라.
9. 후보지 6을 선택하려면 후보지 4도 선택해야 하는 조건을 쓰라.
10. 동부권 최소 하나 선택 조건을 쓰라.
11. 오답 진단: `Maximize 5.0(x1+x3)+7.5(x1+x2)+...`처럼 coverage count에 인구를 곱하는 잘못된 모형이 왜 중복계산을 일으키는지 설명하라.
12. set covering 문제와 max coverage 문제의 차이를 설명하라.

node_id: `n_DM_PDF08.benefit_max_variant`, `n_DM_PDF08.wrong_model_diagnosis`, `n_DM_PDF08.coverage_matrix`, `n_DM_PDF08.binary_open_variable`, `n_DM_PDF04.policy_logic_constraints`

source anchors: `DM_PDF08:p010:L002`, `DM_PDF08:p011:L002`, `DM_PDF08:p012:L002`, `DM_PDF08:p002:L007`, `DM_PDF08:p006:L006`, `DM_PDF04:p021:L002`


In [ ]:
# 힌트:
# - x_j = 후보지 j에 시설을 설치하면 1, 아니면 0이다.
# - A_ij = 수요구역 i가 후보지 j로 커버되면 1, 아니면 0이다.
# - set covering은 각 i에 대해 sum_j A_ij*x_j >= 1이다.
# - set partitioning은 각 i에 대해 sum_j A_ij*x_j = 1이다.
# - set packing은 각 i에 대해 sum_j A_ij*x_j <= 1이다.
# - max coverage에서는 시설 설치 변수 x_j와 혜택 여부 변수 y_i를 분리한다.
# - y_i <= sum_j A_ij*x_j 로 중복계산을 방지한다.


#### 문항별 시각화 학습자료 — 문제 2
- 목적: max coverage에서 설치 변수 `x_j`와 혜택 변수 `y_i`를 분리해야 하는 이유를 본다.
- 체크포인트: coverage count가 2 이상이어도 주민은 한 번만 혜택으로 계산한다.
- 풀이용이므로 최종 운영 조합과 최종 혜택 인구는 표시하지 않는다.


In [ ]:
# 시각화 업데이트: 문제 2 max coverage 중복계산 scaffold
import numpy as np
import matplotlib.pyplot as plt

A = np.array([
    [1,0,1,0,0,0],
    [1,1,0,0,0,0],
    [0,1,1,0,1,0],
    [0,0,1,1,0,0],
    [0,0,0,1,1,1],
    [0,1,0,0,1,0],
    [0,0,0,1,0,1],
    [1,0,0,0,0,1],
])
pop = np.array([5.0,7.5,4.0,6.5,8.0,3.5,9.0,4.5])
# 임의 예시 조합이다. 최적해 판정은 직접 수행한다.
trial_x = np.array([1,0,1,0,0,0])
count = A @ trial_x
y = (count >= 1).astype(int)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].imshow(A, cmap="Blues", vmin=0, vmax=1)
axes[0].set_xticks(range(6), labels=[f"C{j}" for j in range(1,7)], rotation=45)
axes[0].set_yticks(range(8), labels=[f"Zone{i}" for i in range(1,9)])
axes[0].set_title("A_ij: candidate covers zone?")
for i in range(8):
    for j in range(6):
        axes[0].text(j, i, str(A[i,j]), ha="center", va="center", fontsize=8)

axes[1].bar(range(1,9), count, color="#f6b26b")
axes[1].set_title("coverage count = Ax")
axes[1].set_xlabel("zone")
axes[1].set_ylabel("coverage count")

axes[2].bar(range(1,9), pop, color=["#6aa84f" if yy else "#d9d9d9" for yy in y])
axes[2].set_title("benefit y_i counts once")
axes[2].set_xlabel("zone")
axes[2].set_ylabel("population")
plt.tight_layout()
plt.show()

print("trial_x:", trial_x.tolist())
print("coverage count:", count.tolist())
print("y_i:", y.tolist())
print("체크: sum(pop*count)는 중복계산이고, sum(pop*y)는 중복 없는 혜택 계산이다.")


### 내 답안
- 변수 정의:
- 목적함수:
- 제약식:
- domain:
- Solver mapping:
- LP relaxation/논리 해석:
- 오답 진단:
- source anchor / node_id:


### 문제 3. 소형 소방거점 입지와 배정 — capacitated facility location

한 도시는 소형 소방거점 후보지 4곳 중 일부를 열고, 6개 수요구역을 열린 거점에 배정하려고 한다. 각 수요구역은 정확히 하나의 열린 거점에 배정되어야 한다. 후보지별 처리용량과 고정비가 있다.

수요:

| 구역 i | 1 | 2 | 3 | 4 | 5 | 6 |
|---:|---:|---:|---:|---:|---:|---:|
| 수요량 | 20 | 25 | 15 | 30 | 18 | 22 |

후보지:

| 후보지 j | 고정비 | 용량 |
|---:|---:|---:|
| 1 | 100 | 60 |
| 2 | 85 | 55 |
| 3 | 90 | 70 |
| 4 | 70 | 45 |

배정 비용:

| 구역/후보지 | 1 | 2 | 3 | 4 |
|---:|---:|---:|---:|---:|
| 1 | 6 | 9 | 5 | 8 |
| 2 | 4 | 7 | 8 | 6 |
| 3 | 9 | 5 | 6 | 7 |
| 4 | 8 | 6 | 4 | 9 |
| 5 | 7 | 8 | 5 | 6 |
| 6 | 5 | 6 | 7 | 4 |

설치 가능성 제한:

| 구역/후보지 | 1 | 2 | 3 | 4 |
|---:|---:|---:|---:|---:|
| 1 | 1 | 0 | 1 | 1 |
| 2 | 1 | 1 | 0 | 1 |
| 3 | 0 | 1 | 1 | 0 |
| 4 | 1 | 1 | 1 | 0 |
| 5 | 0 | 0 | 1 | 1 |
| 6 | 1 | 0 | 0 | 1 |

요구:
1. 후보지 개방 변수 `y_j`를 정의하라.
2. 구역 i를 후보지 j에 배정하는 변수 `x_ij`를 정의하라.
3. objective function을 고정비와 배정비로 나누어 쓰라.
4. 각 수요구역이 정확히 하나의 후보지에 배정되도록 제약을 쓰라.
5. 열린 후보지에만 배정되도록 `x_ij <= y_j`를 쓰라.
6. 설치 가능성 제한이 0인 위치에는 배정할 수 없도록 제약을 쓰라.
7. 후보지 용량 제약 `sum_i demand_i*x_ij <= capacity_j*y_j`를 쓰라.
8. 최소 2개 이상의 후보지를 열어야 한다는 조건을 추가하라.
9. 이 모형이 set covering, assignment, facility location, fixed-charge 중 어떤 성격을 동시에 가지는지 설명하라.
10. 오답 진단: “각 구역이 커버되기만 하면 되므로 `Ax>=1`만 쓰면 배정문제도 해결된다.” 왜 부족한지 설명하라.
11. Solver에서 changing cells를 `x_ij` matrix와 `y_j` vector로 나누어 설명하라.
12. branch-and-bound가 이 문제를 푸는 데 왜 필요한지 설명하라.

node_id: `n_DM_PDF08.facility_location`, `n_DM_PDF08.coverage_matrix`, `n_DM_PDF08.set_covering_model`, `n_DM_PDF04.fixed_charge_model`, `n_DM_PDF06.assignment_problem`, `n_DM_PDF03.branch_and_bound`

source anchors: `DM_PDF08:p001:L002`, `DM_PDF08:p002:L003`, `DM_PDF08:p002:L007`, `DM_PDF08:p006:L006`, `DM_PDF04:p025:L002`, `DM_PDF04:p026:L002`, `DM_PDF03:p001:L001`


In [ ]:
# 힌트:
# - x_j = 후보지 j에 시설을 설치하면 1, 아니면 0이다.
# - A_ij = 수요구역 i가 후보지 j로 커버되면 1, 아니면 0이다.
# - set covering은 각 i에 대해 sum_j A_ij*x_j >= 1이다.
# - set partitioning은 각 i에 대해 sum_j A_ij*x_j = 1이다.
# - set packing은 각 i에 대해 sum_j A_ij*x_j <= 1이다.
# - max coverage에서는 시설 설치 변수 x_j와 혜택 여부 변수 y_i를 분리한다.
# - y_i <= sum_j A_ij*x_j 로 중복계산을 방지한다.


#### 문항별 시각화 학습자료 — 문제 3
- 목적: 배정 가능성, 배정비, 후보지 용량을 분리해서 본다.
- 체크포인트: `Ax>=1`만으로는 정확히 하나 배정, open 연결, capacity를 표현할 수 없다.
- 풀이용이므로 최종 개방 조합과 최종 배정은 표시하지 않는다.


In [ ]:
# 시각화 업데이트: 문제 3 facility location 입력 구조 scaffold
import numpy as np
import matplotlib.pyplot as plt

D = np.array([20,25,15,30,18,22])
fixed = np.array([100,85,90,70])
cap = np.array([60,55,70,45])
cost = np.array([
    [6,9,5,8],
    [4,7,8,6],
    [9,5,6,7],
    [8,6,4,9],
    [7,8,5,6],
    [5,6,7,4],
])
avail = np.array([
    [1,0,1,1],
    [1,1,0,1],
    [0,1,1,0],
    [1,1,1,0],
    [0,0,1,1],
    [1,0,0,1],
])

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(avail, cmap="Greens", vmin=0, vmax=1)
axes[0].set_xticks(range(4), labels=[f"C{j}" for j in range(1,5)])
axes[0].set_yticks(range(6), labels=[f"Zone{i}" for i in range(1,7)])
axes[0].set_title("availability a_ij")
for i in range(6):
    for j in range(4):
        axes[0].text(j, i, str(avail[i,j]), ha="center", va="center")

im = axes[1].imshow(cost, cmap="Oranges")
axes[1].set_xticks(range(4), labels=[f"C{j}" for j in range(1,5)])
axes[1].set_yticks(range(6), labels=[f"Zone{i}" for i in range(1,7)])
axes[1].set_title("assignment unit cost c_ij")
for i in range(6):
    for j in range(4):
        axes[1].text(j, i, str(cost[i,j]), ha="center", va="center", fontsize=8)

x = np.arange(4)
axes[2].bar(x-0.15, cap, width=0.3, label="capacity", color="#6fa8dc")
axes[2].bar(x+0.15, fixed, width=0.3, label="fixed cost", color="#f6b26b")
axes[2].set_xticks(x, labels=[f"C{j}" for j in range(1,5)])
axes[2].set_title("capacity and fixed cost by candidate")
axes[2].legend(fontsize=8)
plt.tight_layout()
plt.show()

print("체크: x_ij <= y_j, sum_i d_i*x_ij <= capacity_j*y_j 를 함께 써야 한다.")


### 내 답안
- 변수 정의:
- 목적함수:
- 제약식:
- domain:
- Solver mapping:
- LP relaxation/논리 해석:
- 오답 진단:
- source anchor / node_id:
